# 真实输入驱动的科研 Notebook 模板（R）
与 Python 模板使用同一份 `input_config.json`、同一输出结构和验收字段。需要 IRkernel（自带 jsonlite、digest 依赖），其余仅用基础 R。
本模板以分组统计展示接入流程，实际论文复现须替换为已核验的方法及基线。R 内核不提供可靠的已执行源码记录，`execution:*` 保持 NOT-RUN：交付前请重启内核并 Run All。

## 1. 科学输入与可追溯配置

科学参数集中在此。所有未标记 `plot` 的代码单元都计入计算代码身份。

In [ ]:
source("notebook_provenance.R", encoding = "UTF-8")

INPUT_CONFIG <- "input_config.json"
NOTEBOOK_FILE <- "notebook_template_r.ipynb"  # 重命名 Notebook 后同步修改
LOCAL_CODE <- c("notebook_provenance.R")      # 参与计算的本地 R 文件，逐个登记
OUTPUT_DIR <- "example_output_r"
METHOD <- "group-summary"
PARAMETERS <- list(group_column = "group", value_column = "value", ddof = 1,
                   missing_policy = "exclude-empty-group-or-nonfinite-value")

config <- read_config(INPUT_CONFIG)
filled <- function(x) !is.null(x) && length(x) > 0 && !identical(x, "")
gaps <- Filter(function(key) !filled(config[[key]]),
               c("dataset_id", "path", "source", "source_type", "role", "sha256", "columns", "units"))
if (identical(config$dataset_id, "YOUR_DATASET_VERSION")) gaps <- c(gaps, "dataset_id (placeholder)")
if (is.null(config$missing_values)) gaps <- c(gaps, "missing_values")
if (!filled(config$units$value)) gaps <- c(gaps, "units.value")
if (!filled(config$columns$group) || !filled(config$columns$value)) gaps <- c(gaps, "columns.group/value")
if (filled(config$source_type) && !(config$source_type %in% c("user_data", "paper_data", "public_observations"))) gaps <- c(gaps, "source_type (unknown value)")
if (filled(config$role) && !(config$role %in% c("paper_reproduction", "method_transfer", "input_example"))) gaps <- c(gaps, "role (unknown value)")
if (identical(config$role, "paper_reproduction") && !filled(config$paper_reference)) gaps <- c(gaps, "paper_reference")
if (!filled(config$columns$sample_id) && !identical(config$id_policy, "source_row_number")) gaps <- c(gaps, "id_policy")
keys <- if (filled(config$key_columns)) config$key_columns else if (filled(config$columns$sample_id)) config$columns$sample_id
if (filled(config$key_columns) && !(config$columns$sample_id %in% keys)) gaps <- c(gaps, "key_columns (must include sample_id)")
DATA_FILE <- if (filled(config$path) && grepl("^(/|~|[A-Za-z]:)", config$path)) path.expand(config$path) else file.path(dirname(INPUT_CONFIG), config$path)
if (!length(DATA_FILE) || !file.exists(DATA_FILE)) gaps <- c(gaps, "path (input file missing, no simulated fallback)")
if (length(gaps)) stop("BLOCKED: configure ", paste(gaps, collapse = ", "), call. = FALSE)

dir.create(OUTPUT_DIR, showWarnings = FALSE, recursive = TRUE)
checks <- new_checks(file.path(OUTPUT_DIR, "validation_matrix.csv"), config$dataset_id, METHOD,
                     paste0("notebook-sha256:", science_hash(NOTEBOOK_FILE)),
                     c("input:availability", "input:identity", "input:schema", "input:units", "input:role", "input:qc",
                       "input:preview", "execution:source-match", "execution:clean-kernel"))
check(checks, "input:availability", file.exists(DATA_FILE), "Input file must exist")
check(checks, "input:identity", sha256(DATA_FILE) == config$sha256, "BLOCKED: input SHA-256 changed; verify dataset version")
check(checks, "input:units", filled(config$units$value), "Value unit must be declared")
check(checks, "input:role", filled(config$role), "Scientific role must be declared")

## 2. 输入检查与 QC

全部字段按文本读入，不让 R 自动识别 NA 或转换类型；逐条保存排除原因。

In [ ]:
encoding <- if (is.null(config$encoding) || tolower(config$encoding) == "utf-8-sig") "UTF-8-BOM" else config$encoding
separator <- if (!is.null(config$delimiter)) config$delimiter else if (grepl("\\.tsv$", DATA_FILE, ignore.case = TRUE)) "\t" else ","
original <- read_text_table(DATA_FILE, separator, encoding)
needed <- c(config$columns$group, config$columns$value, keys)
check(checks, "input:schema", all(needed %in% names(original)) && !anyDuplicated(names(original)) && nrow(original) > 0 &&
        (is.null(config$expected_rows) || nrow(original) == config$expected_rows), "Required columns and expected record count")
raw <- data.frame(sample_id = if (filled(config$columns$sample_id)) original[[config$columns$sample_id]] else paste0("row:", seq_len(nrow(original))),
                  group_raw = original[[config$columns$group]], value_raw = original[[config$columns$value]], stringsAsFactors = FALSE)
key_parts <- if (length(keys)) original[keys] else data.frame(source_row = as.character(seq_len(nrow(original))))
raw$sample_key <- vapply(seq_len(nrow(raw)), function(i) json_key(as.list(key_parts[i, , drop = FALSE])), "")
check(checks, "input:schema", !anyDuplicated(raw$sample_key) && all(trimws(unlist(key_parts)) != ""),
      "BLOCKED: blank or duplicate key; for repeated measurements configure key_columns")
raw$group <- ifelse(raw$group_raw %in% config$missing_values, "", raw$group_raw)
raw$value <- ifelse(raw$value_raw %in% config$missing_values, "", raw$value_raw)

text <- trimws(raw$value)
is_number <- grepl("^[+-]?([0-9]+\\.?[0-9]*|\\.[0-9]+)([eE][+-]?[0-9]+)?$", text)
values <- rep(NA_real_, nrow(raw))
values[is_number] <- as.numeric(text[is_number])
declared_missing <- raw$value_raw %in% config$missing_values
nonfinite <- !declared_missing & (tolower(text) %in% c("nan", "inf", "+inf", "-inf", "infinity", "-infinity") | (is_number & !is.finite(values)))
undeclared_blank <- !declared_missing & text == ""
reasons <- list(empty_group = trimws(raw$group) == "", declared_missing = declared_missing, undeclared_blank = undeclared_blank,
                nonfinite_value = nonfinite, unparseable_value = !declared_missing & !nonfinite & !undeclared_blank & !is_number)
invalid <- Reduce(`|`, reasons)
excluded <- raw[invalid, ]
excluded$qc_reason <- vapply(which(invalid), function(i) paste(names(reasons)[vapply(reasons, `[`, NA, i)], collapse = ";"), "")
write_table(excluded, file.path(OUTPUT_DIR, "excluded_records.csv"))
valid <- raw[!invalid, ]
valid$value <- values[!invalid]
qc_counts <- lapply(reasons, sum)
check(checks, "input:qc", nrow(valid) > 0 && all(is.finite(valid$value)), "Valid observations remain; every exclusion has a reason")
cat(sprintf("Input=%d, valid=%d, excluded=%d\n", nrow(raw), nrow(valid), nrow(excluded)))
if (length(unique(raw$sample_id)) < nrow(raw)) cat("REVIEW: repeated measurements per sample_id; records are not independent replicates.\n")
head(valid, 5)

## 3. 核心计算、保存与来源记录

示例仅演示分组均值、样本 SD（n-1），不代表任何领域算法验证。单记录组的 SD 无定义，保留缺失。

In [ ]:
# radix 排序按字节序，不受系统 locale 影响。
groups <- sort(unique(valid$group), method = "radix")
summary_table <- do.call(rbind, lapply(groups, function(g) {
  x <- valid$value[valid$group == g]
  data.frame(group = g, n = length(x), n_units = length(unique(valid$sample_id[valid$group == g])),
             mean = mean(x), sd = if (length(x) > 1) sd(x) else NA_real_, stringsAsFactors = FALSE)
}))
stopifnot(sum(summary_table$n) == nrow(valid), all(is.finite(summary_table$mean)), PARAMETERS$ddof == 1)
if (sha256(DATA_FILE) != config$sha256) stop("Input changed during the run")
summary_path <- file.path(OUTPUT_DIR, "group_summary.csv")
observations_path <- file.path(OUTPUT_DIR, "observations_for_plot.csv")
write_table(summary_table, summary_path)
write_table(valid[c("sample_id", "sample_key", "group", "value")], observations_path)
calculation <- list(
  schema_version = "1.1", synthetic_example = FALSE, method = METHOD,
  input = list(dataset_id = config$dataset_id, source = config$source, role = config$role, units = config$units,
               columns = config$columns, file = basename(DATA_FILE), sha256 = sha256(DATA_FILE), config_sha256 = sha256(INPUT_CONFIG)),
  created_utc = format(Sys.time(), "%Y-%m-%dT%H:%M:%SZ", tz = "UTC"),
  scientific_parameters = PARAMETERS, science_code_sha256 = science_hash(NOTEBOOK_FILE),
  local_code_sha256 = setNames(lapply(LOCAL_CODE, sha256), LOCAL_CODE),
  environment = list(r = R.version.string, platform = R.version$platform),
  result = list(path = basename(summary_path), sha256 = sha256(summary_path), rows = nrow(summary_table),
                observations = nrow(valid), observations_path = basename(observations_path), observations_sha256 = sha256(observations_path)),
  qc = list(input_rows = nrow(raw), excluded_rows = nrow(excluded), reasons = qc_counts))
write_record(list(calculation = calculation, figures = setNames(list(), character(0))), file.path(OUTPUT_DIR, "provenance.json"))
set_check(checks, "execution:source-match", "NOT-RUN", "R kernel keeps no reliable executed source; save, restart and Run All")
set_check(checks, "execution:clean-kernel", "NOT-RUN", "Record after an actual restart and Run All")
save_checks(checks)
summary_table

## 4. 独立绘图：干净 Kernel 只运行下一单元

身份不匹配即停止。基础 R 图形设备无法可靠报告缺字形与越界，对应检查保持 NOT-RUN，须打开图件人工确认。

In [ ]:
source("notebook_provenance.R", encoding = "UTF-8")

INPUT_CONFIG <- "input_config.json"
NOTEBOOK_FILE <- "notebook_template_r.ipynb"
OUTPUT_DIR <- "example_output_r"
EXPECTED_PARAMETERS <- list(group_column = "group", value_column = "value", ddof = 1,
                            missing_policy = "exclude-empty-group-or-nonfinite-value")
# ===== 只影响图形的参数 =====
WIDTH_MM <- 180; HEIGHT_MM <- 110  # 可修改试稿尺寸，并非通用期刊要求
FONT_SIZE <- 10; DPI <- 300
TITLE <- "Group mean \u00b1 sample SD"
COLOR <- "#4878A8"
FONT_FAMILY <- ""  # 中文标签改为已安装字体，如 "PingFang SC" 或 "Noto Sans CJK SC"
FORMAT <- "png"  # 每图一种格式：png（默认）/ tiff；目标规范明确要求矢量图时用 pdf
stopifnot(FORMAT %in% c("png", "tiff", "pdf"))

record_path <- file.path(OUTPUT_DIR, "provenance.json")
record <- read_record(record_path)
calculation <- record$calculation
result_path <- file.path(OUTPUT_DIR, calculation$result$path)
observations_path <- file.path(OUTPUT_DIR, calculation$result$observations_path)
checks <- new_checks(file.path(OUTPUT_DIR, "validation_matrix.csv"), calculation$input$dataset_id, calculation$method,
                     paste0("notebook-sha256:", science_hash(NOTEBOOK_FILE)),
                     c("provenance:result", "provenance:parameters", "provenance:code", "input:identity", "figure:nonempty",
                       "figure:valid-values", "figure:data-rows", "figure:no-glyph-warning", "figure:within-canvas", "figure:visual-check"))
check(checks, "provenance:result", sha256(result_path) == calculation$result$sha256 &&
        sha256(observations_path) == calculation$result$observations_sha256, "Result changed; stop and investigate")
check(checks, "provenance:parameters", isTRUE(all.equal(EXPECTED_PARAMETERS, calculation$scientific_parameters)),
      "Science parameters changed; explicitly recompute")
unchanged <- vapply(names(calculation$local_code_sha256), function(name) file.exists(name) && sha256(name) == calculation$local_code_sha256[[name]], NA)
check(checks, "provenance:code", science_hash(NOTEBOOK_FILE) == calculation$science_code_sha256 && all(unchanged),
      "Science code changed; explicitly recompute")
input_file <- character(0)
if (file.exists(INPUT_CONFIG)) {
  check(checks, "input:identity", sha256(INPUT_CONFIG) == calculation$input$config_sha256, "Input config changed; explicitly recompute")
  configured <- read_config(INPUT_CONFIG)$path
  input_file <- if (grepl("^(/|~|[A-Za-z]:)", configured)) path.expand(configured) else file.path(dirname(INPUT_CONFIG), configured)
}
input_verified <- length(input_file) == 1 && file.exists(input_file)
if (input_verified) {
  check(checks, "input:identity", sha256(input_file) == calculation$input$sha256, "Input changed; explicitly recompute")
} else {
  set_check(checks, "input:identity", "BLOCKED", "Archived result: original input unavailable, not revalidated")
  cat("BLOCKED input:identity - archived result; original input NOT revalidated.\n")
}

plot_data <- read_text_table(result_path)
plot_data[c("n", "n_units", "mean", "sd")] <- lapply(plot_data[c("n", "n_units", "mean", "sd")], as.numeric)
observations <- read_text_table(observations_path)
observations$value <- as.numeric(observations$value)
check(checks, "figure:nonempty", nrow(plot_data) > 0 && nrow(observations) > 0, "No data to plot")
check(checks, "figure:valid-values", all(is.finite(plot_data$mean)) && all(is.finite(observations$value)) &&
        all(is.na(plot_data$sd) == (plot_data$n == 1)), "Plotted values must be finite; SD undefined only for n=1")
check(checks, "figure:data-rows", nrow(plot_data) == calculation$result$rows && nrow(observations) == calculation$result$observations &&
        identical(as.numeric(table(factor(observations$group, plot_data$group))), plot_data$n), "Unexpected plot row or group counts")

figure_file <- file.path(OUTPUT_DIR, paste0("group_mean_example.", FORMAT))
draw <- function() {
  par(mar = c(6, 5, 3, 1), las = 1, family = FONT_FAMILY)
  x <- seq_len(nrow(plot_data))
  low <- pmin(plot_data$mean - ifelse(is.na(plot_data$sd), 0, plot_data$sd), min(observations$value))
  high <- pmax(plot_data$mean + ifelse(is.na(plot_data$sd), 0, plot_data$sd), max(observations$value))
  plot(NA, xlim = range(x) + c(-0.5, 0.5), ylim = range(low, high), xaxt = "n", xlab = "",
       ylab = sprintf("%s (%s)", calculation$input$columns$value, calculation$input$units$value), main = TITLE, font.main = 1)
  # 横向偏移仅防止点重叠，不改变测量值。
  for (i in x) {
    y <- observations$value[observations$group == plot_data$group[i]]
    offset <- if (length(y) > 1) seq(-0.14, 0.14, length.out = length(y)) else 0
    points(i + offset, y, pch = 16, cex = 0.6, col = adjustcolor(COLOR, 0.3))
  }
  ok <- !is.na(plot_data$sd) & plot_data$sd > 0
  if (any(ok)) arrows(x[ok], (plot_data$mean - plot_data$sd)[ok], x[ok], (plot_data$mean + plot_data$sd)[ok], angle = 90, code = 3, length = 0.05, lwd = 1.5)
  points(x, plot_data$mean, pch = 18, cex = 1.4)
  repeated <- plot_data$n_units < plot_data$n
  axis(1, x, labels = FALSE)
  mtext(plot_data$group, side = 1, line = 1, at = x)
  mtext(sprintf("(n=%d%s)", as.integer(plot_data$n), ifelse(repeated, sprintf(" records, %d units", as.integer(plot_data$n_units)), "")),
        side = 1, line = 2, at = x)
  mtext(ifelse(plot_data$n == 1, "SD undefined", ""), side = 1, line = 3, at = x)
  title(xlab = calculation$input$columns$group, line = 4.5)
}
# 写入临时文件，成功后再替换，失败的绘图不会留下看似完成的图件。
temporary <- paste0(figure_file, ".tmp")
# macOS 的 quartz 设备会忽略 LZW 并写出 RGBA，所以 TIFF 固定用 cairo。
switch(FORMAT,
  png = png(temporary, WIDTH_MM, HEIGHT_MM, units = "mm", res = DPI, pointsize = FONT_SIZE),
  tiff = tiff(temporary, WIDTH_MM, HEIGHT_MM, units = "mm", res = DPI, pointsize = FONT_SIZE, compression = "lzw", type = "cairo"),
  pdf = pdf(temporary, WIDTH_MM / 25.4, HEIGHT_MM / 25.4, pointsize = FONT_SIZE))
tryCatch(draw(), finally = dev.off())
if (!file.exists(temporary)) {
  stop("BLOCKED: the ", FORMAT, " device wrote nothing; LZW TIFF needs a working cairo (XQuartz on macOS). Use png or pdf", call. = FALSE)
}
if (!file.rename(temporary, figure_file)) stop("cannot write ", figure_file)

record$figures$group_mean_example <- list(
  file = basename(figure_file), sha256 = sha256(figure_file), result_sha256 = calculation$result$sha256, input_revalidated = input_verified,
  style = list(width_mm = WIDTH_MM, height_mm = HEIGHT_MM, font_size = FONT_SIZE, dpi = DPI, title = TITLE, color = COLOR, font_family = FONT_FAMILY, format = FORMAT))
write_record(record, record_path)
set_check(checks, "figure:visual-check", "NOT-RUN", paste("Open", basename(figure_file), "- labels, glyphs, clipping, legibility"))
save_checks(checks)

## 5. 交付边界

输出结构与 Python 模板一致，可用审计技能的 `compare_csv.py` 按 `group` 对照两种语言的 `group_summary.csv`。检查状态不代表论文方法复现通过；NOT-RUN 项须实际查看后再填写。